In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

In [ ]:
# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# 4. Print shape of one batch
images, labels = next(iter(train_loader))
print(f'Training Batch Input Size: {images.shape}')
print(f'Training Batch Labels Size: {labels.shape}')

In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)

    # Convert from (C, H, W) to (H, W, C) for matplotlib
    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Task 1: Write your model class here:

import torch.nn as nn


class NN4Layer(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super().__init__()

    self.Layer1 = nn.Linear(input_dim, hidden_dim)
    self.Layer2 = nn.Linear(hidden_dim, hidden_dim)
    self.Layer3 = nn.Linear(hidden_dim, hidden_dim)
    self.Layer4 = nn.Linear(hidden_dim, output_dim)

    self.relu = nn.ReLU()

  def forward(self, x):
    a1 = self.relu(self.Layer1(x))
    a2 = self.relu(self.Layer2(a1))
    a3 = self.relu(self.Layer3(a2))
    output = self.Layer4(a3)

    return output

In [ ]:
# Task 2: Write your training loop here:
from tqdm import tqdm
def train_one_epoch(model, optimizer, criterion, train_loader, device):
  model.train()
  tot_loss = 0

  for xb, yb in train_loader:
    xb = xb.view(xb.size(0), -1).to(device)
    yb = yb.view(-1, 1).to(device)

    output = model(xb)
    loss = criterion(output, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    tot_loss += loss.item()

  avg = tot_loss/ len(train_loader)
  return avg

In [ ]:
# Task 3: Write your validation loop here:

def validate(model, criterion, test_loader, device):
  model.eval()
  tot_loss = 0

  with torch.no_grad():
    for xb, yb in test_loader:
      xb = xb.view(xb.size(0), -1).to(device)
      yb = yb.view(-1, 1).to(device)

      print(xb.size)

      output = model(xb)
      loss = criterion(output, yb)
      tot_loss += loss.item()

  avg = tot_loss/len(test_loader)
  return avg

In [ ]:
# Task 4: Define device, model, loss, optimizer:

from torch.optim import AdamW


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


print(f'Input dim (channels, height, width): {images[0].size()}') #(3, 36, 36)

inputD = 3 * 36 * 36
hiddenD = 128
outputD = 1 #regression models ouputs a single number

model = NN4Layer(inputD, hiddenD, outputD)
model.to(device)


criterion = nn.MSELoss()

optimizer = AdamW(model.parameters(), 1e-3)

In [ ]:
# Task 5: Start training for 20 epochs:

num_epochs = 20
train_losses = []
val_losses = []

print('Start Training ---')
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)
    val_loss = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print('-'*30)
    print(f'\nEpoch {epoch+1}/{num_epochs} ||   Train loss: {train_loss:4f},   Val Loss: {val_loss:.4f}\n')
    print('-'*30)

print('Training Completed!!')

In [ ]:
# Task 1: Write your code here:
# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
#The loss plot seem so noisy we need to fix it so it gets smother

In [ ]:
# @title
# Task 2 (Bonus): Write your code here:

#something went off
'''
model.eval()

test_images, test_labels = next(iter(test_loader))

#
with torch.no_grad():
  predictLabel = model(test_images.view(test_images.size(0), -1).to(device))

# Move to CPU for plotting
test_images = test_images.cpu()
predictLabel = predictLabel.cpu()

# Plot original vs reconstructed
n_images = 9
fig, axes = plt.subplots(2, n_images, figsize=(18, 6))

for i in range(n_images):
  # Original
  axes[0, i].imshow(test_images[i], cmap='gray')
  axes[0, i].axis('off')
  axes[0, i].set_title(f'Original - "{test_labels[i]}"', fontsize=12)

  # Reconstructed
  axes[1, i].imshow(predictLabel[i].squeeze(), cmap='gray')
  axes[1, i].axis('off')
  axes[1, i].set_title(f'Reconstructed - "{test_labels[i]}"', fontsize=12)

plt.suptitle('Original vs Reconstructed Images', fontsize=14)
plt.tight_layout()
plt.show()
'''
print()